In [1]:
import glob
import re
import os

import numpy as np
from pathlib import Path

from pymor.basic import *
from pymor.core.pickle import load

from RBInvParam.problems.elasticity.build import build_InstationaryModelIP

set_log_levels({
    'pymor' : 'WARN'
})

set_defaults({})


In [2]:
import matplotlib as mpl
import matplotlib.pyplot as plt

fontsize = 14
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "cm",
    "font.size": fontsize,
    #'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{accents}',
    'figure.dpi': 200
})

Exception in thread Control:
Traceback (most recent call last):
  File "/home/dealii/workdir/venv/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/home/dealii/workdir/venv/lib/python3.10/site-packages/ipykernel/kernelbase.py", line 339, in dispatch_control
    await self.process_control(msg)
  File "/home/dealii/workdir/venv/lib/python3.10/site-packages/ipykernel/kernelbase.py", line 352, in process_control
    self.log.debug("Control received: %s", msg)
  File "/usr/lib/python3.10/logging/__init__.py", line 1465, in debug
    self._log(DEBUG, msg, args, **kwargs)
  File "/usr/lib/python3.10/logging/__init__.py", line 1624, in _log
    self.handle(record)
  File "/usr/lib/python3.10/logging/__init__.py", line 1634, in handle
    self.callHandlers(record)
  File "/usr/lib/python3.10/logging/__init__.py", line 1696, in callHandlers
    hdlr.handle(record)
  File "/usr/lib/python3.10/logging/__init__.py", line 968, in handle
    sel

In [3]:
from typing import Dict, Tuple, Optional

def get_last_file(path: Path) -> Path | None:
    # --- Step 1: Look for final files first ---
    final_candidates = [
        path / "TR_IRGNM_final.pkl",
        path / "FOM_IRGNM_final.pkl"
    ]
    
    for final_file in final_candidates:
        if final_file.exists():
            return final_file.name  # Return immediately if found
    
    # --- Step 2: If no final file exists, find the highest index file ---
    files = glob.glob(os.path.join(path, "TR_IRGNM_*.pkl"))
    files += glob.glob(os.path.join(path, "FOM_IRGNM_*.pkl"))

    indexed_files = []
    for f in files:
        match = re.search(r'(?:TR|FOM)_IRGNM_(\d+)\.pkl$', os.path.basename(f))
        if match:
            idx = int(match.group(1))
            indexed_files.append((idx, f))

    if indexed_files:
        _, max_file = max(indexed_files, key=lambda x: x[0])
        return Path(max_file).name

    print("No matching IRGNM result files found.")
    return None

def filter_and_reorder(d: Dict, pattern: str = r'.*FOM.*') -> Tuple[Dict, Optional[str]]:
    regex = re.compile(pattern)
    matching = [k for k in d if regex.search(k)]
    non_matching = [k for k in d if k not in matching]

    # Sort keys alphabetically within each group
    non_matching_sorted = sorted(non_matching)
    matching_sorted = sorted(matching)

    # Build the reordered dict: alphabetically sorted non-matching first, then matching ones
    reordered = {k: d[k] for k in non_matching_sorted}
    reordered.update({k: d[k] for k in matching_sorted})

    # Return reordered dict and the single matching key (if exactly one match)
    if len(matching_sorted) == 1:
        return reordered, matching_sorted[0]
    else:
        return reordered, None

In [7]:
#SAVE_PATH = Path('/home/dealii/workdir/figs')
SAVE_PATH = Path('/home/benedikt/Schreibtisch/error_stagnation')

########################################################################################

#WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')
WORK_DIR = Path('/home/dealii/workdir/experiments')
# #WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')

subset = 'new_baseline'
obs_op = 'grid'
data_dir_path = WORK_DIR / 'new_pre_paper_version' / 'new_baseline_noise_level'

experiment_names = []
pattern = re.compile(rf'^.*_0.0002_.*_{obs_op}')

# subset = 'new_baseline'
# obs_op = 'identity'
# data_dir_path = WORK_DIR / 'pre_paper_version' / 'new_baseline_5_1e-9'

# experiment_names = []
# pattern = re.compile(rf'^.*_{obs_op}')
# #pattern = re.compile(rf'^.*')

experiment_names += [
    d.name for d in data_dir_path.iterdir()
    if d.is_dir() and pattern.match(d.name)
]

data_paths = [data_dir_path / experiment_name for experiment_name in experiment_names]
file_names = [get_last_file(data_path) for data_path in data_paths]


########################################################################################

# data_dir_path = Path('/home/dealii/workdir/examples/elasticity/dumps') / '20251130_142544_TR_IRGNM'

# experiment_names = []
# pattern = re.compile(rf'.*')

# experiment_names += [
#     d.name for d in data_dir_path.iterdir()
#     if d.is_dir() and pattern.match(d.name)
# ]

# data_paths = [data_dir_path]
# file_names = [get_last_file(data_path) for data_path in data_paths]

########################################################################################

setup = None
data = {}
optimizer_parameters = {}

for (data_path, file_name) in zip(data_paths, file_names):            

    try:
        with open(data_path / file_name, 'rb') as file:
            data_ = load(file)
        data[str(data_path.name)] = data_
    except TypeError:
        print(f"Can not find dumps for {data_path}")
    except:
        print(f"Can not open {data_path / file_name}")

    if not setup:
        with open(data_path / 'setup.pkl', 'rb') as file:
            setup = load(file)

    
    optimizer_parameter_path = data_path / 'optimizer_parameter.pkl'
    with open(optimizer_parameter_path, 'rb') as file:
        optimizer_parameter = load(file)
        
    optimizer_parameters[str(data_path.name)] = optimizer_parameter

data, FOM_key = filter_and_reorder(data, pattern=r'.*FOM.*')
#assert FOM_key

if not 'FOM' in locals():
    FOM = build_InstationaryModelIP(setup=setup)

print(data.keys())


Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

[2025-12-19 14:32:22,728][build_InstationaryModelIP] - Construct problem..
	 Setting up BC constraints.
	 Setting up system matrizies.
	 Using CosseratDelamination SystemMatrix
	 Using CenterExcite BodyForce
	 Assembling force list.
	 ---------------------- 
	 #DoFs: 14415
	 #Parameter: 961
[2025-12-19 14:32:33,613][__init__] - Setting up InstationaryModelIP
[2025-12-19 14:32:33,616][reset_cached_operators] - Deleting cache


RuntimeError: Unknown ObservationOperatorType.

In [ ]:
import pandas as pd

rows = []
regex = re.compile('.*FOM.*')

columns = [
    'Algorithm',
    'Variant',
    'Q-err',
    'time [s]', 
    'speed up', 
    'FOM solves', 
    r'$n_Q$', 
    r'$n_V$',
    'o. iter',
    'total iter.'
]

FOM_total_runtime = data[FOM_key]['total_runtime'][-1]
FOM_q = data[FOM_key]['q'][-1]

FOM_row = [
    FOM_key,
    '--',
    np.nan,
    int(FOM_total_runtime),
    '--',
    '--',
    '--',
    '--',
    len(data[FOM_key]['J'])-1,
    len(data[FOM_key]['J'])-1
]

for key, val in data.items():
    if key == FOM_key:
        continue
        
    # FOM_solves = val['FOM_num_calls']['solve_state'] + \
    #              val['FOM_num_calls']['solve_adjoint'] + \
    #              val['FOM_num_calls']['solve_linearized_state'] + \
    #              val['FOM_num_calls']['solve_linearized_adjoint']
    FOM_solves = '--'

    speed_up = FOM_total_runtime / val['total_runtime'][-1]
    TR_Js = []
    inner_loop_statistics = val['inner_loop_statistics']
    for inner_loop_statistic in inner_loop_statistics:
        TR_Js += inner_loop_statistic['J']

    diff = FOM_q - val['q'][-1]
    Q_rel_err = np.sqrt(FOM.products['prod_Q'].pairwise_apply2(diff, diff)[0]) / np.sqrt(FOM.products['prod_Q'].pairwise_apply2(FOM_q, FOM_q)[0])
    
    row = [
        key,
        '--',
        float(Q_rel_err),
        int(val['total_runtime'][-1]),
        speed_up,
        FOM_solves,
        val['dim_Q_r'][-1],
        val['dim_V_r'][-1],
        len(val['J']),
        len(TR_Js)
    ]
        
    rows.append(row)

rows = [FOM_row] + rows
df = pd.DataFrame.from_records(rows, columns=columns)
df['Q-err'] = df['Q-err'].apply(
    lambda x: "{:.2e}".format(x) if not np.isnan(x) else "--"
)

df['Algorithm'] = ['FOM','TR','TR','TR']
df['Variant'] = ['--','I','II','III']
df

In [10]:
#print(df.to_latex(index=False, float_format="%.2f", column_format='lc|cccccccc'))